# EV Charging Network — Coverage Gap Analysis & Priority Scoring
*DATA 552 – Group 4*

**Which Canadian cities should be prioritised for new fast-charger installations?**

It builds on three prior outputs:
- `ev_chargers_geo.csv` – cleaned station-level data
- `ev_vehicle_registrations_raw.csv` – Statistics Canada quarterly EV registration counts
- `location_stats.csv` – city-level merged table with charger counts, EV counts, ratios, and NNI Index

### Approach
1. **Clean** the merged city dataset — handle edge cases (cities with zero EV registrations, missing NNI)
2. **Classify** each city into a coverage tier based on its fast-charger-to-EV ratio
3. **Score** each city with a weighted priority formula  
4. **Rank** cities and export a prioritised list for planning use

**Threshold:** Industry guidance and the team wrangling notebook use **1 fast charger per 300 EVs** as the minimum adequate coverage ratio (≈ 0.00333). Cities below this line are considered under-served.

In [17]:
import os
import pandas as pd
import numpy as np
import altair as alt
import warnings
warnings.filterwarnings("ignore")
os.makedirs("docs/img", exist_ok=True)
df_raw = pd.read_csv("location_stats.csv")
df_raw.head()

,location,ev_dc_fast_count,ev_level2_evse_num,vehicle count,fast charger ratio,level 2 charger ratio,total ratio,NNI Index
0,Abbotsford,24,0.0,11360.0,0.002113,0.000000,0.002113,1.189378
1,Ajax,4,3.0,5846.0,0.000684,0.000513,0.001197,2.489261
2,Arnprior,2,1.0,344.0,0.005814,0.002907,0.008721,6.308889
3,Ashcroft,2,2.0,0.0,inf,inf,inf,NaN
4,Atikokan,2,0.0,10.0,0.200000,0.000000,0.200000,NaN


## Step 1: Data Cleaning

In [21]:
df = df_raw.copy()

n_inf = np.isinf(df[["fast charger ratio", "total ratio"]]).sum().sum()
df.replace([np.inf, -np.inf], np.nan, inplace=True)

n_nni_nan = df["NNI Index"].isna().sum()
df["NNI Index"] = df["NNI Index"].fillna(1.0)

df_scored = df.dropna(subset=["vehicle count", "fast charger ratio"]).copy()
df_scored = df_scored[df_scored["vehicle count"] > 0].copy()

df_no_vehicles = df[df["vehicle count"].isna() | (df["vehicle count"] == 0)]

print(f"\nCities with EV registration data (scoreable): {len(df_scored)}")
print(f"Cities lacking vehicle registration match: {len(df_no_vehicles)}")
df_scored.head()


Cities with EV registration data (scoreable): 231
Cities lacking vehicle registration match: 22



Cities with EV registration data (scoreable): 231
Cities lacking vehicle registration match: 22


,location,ev_dc_fast_count,ev_level2_evse_num,vehicle count,fast charger ratio,level 2 charger ratio,total ratio,NNI Index
0,Abbotsford,24,0.0,11360.0,0.002113,0.000000,0.002113,1.189378
1,Ajax,4,3.0,5846.0,0.000684,0.000513,0.001197,2.489261
2,Arnprior,2,1.0,344.0,0.005814,0.002907,0.008721,6.308889
4,Atikokan,2,0.0,10.0,0.200000,0.000000,0.200000,1.000000
5,Aurora,2,0.0,4502.0,0.000444,0.000000,0.000444,1.000000


## Step 2: Coverage Tier Classification

Each city is assigned to one of three tiers based on its DC fast charger–to–EV ratio:

- Under-served: ratio < 1/300 (0.00333)
- Near Threshold: 1/300 <= ratio < 1/150 (0.00667)
- Well-served: ratio >= 1/150

Note: the 1/300 threshold comes from team research and industry guidelines cited in the wrangling notebook.

In [51]:
THRESHOLD_MIN = 1 / 300
THRESHOLD_OK  = 1 / 150

def assign_tier(ratio):
    if pd.isna(ratio):
        return "Unknown"
    elif ratio < THRESHOLD_MIN:
        return "Under-served"
    elif ratio < THRESHOLD_OK:
        return "Near Threshold"
    else:
        return "Well-served"

df_scored["tier"] = df_scored["fast charger ratio"].apply(assign_tier)

tier_summary = df_scored["tier"].value_counts().reindex(
    ["Under-served", "Near Threshold", "Well-served"]
)
tier_pct = tier_summary / tier_summary.sum() * 100

print("Coverage tier:")
for tier, count in tier_summary.items():
    print(f"{tier}: {count} cities ({tier_pct[tier]:.1f}%)")

tier_ev = df_scored.groupby("tier")["vehicle count"].sum().reindex(
    ["Under-served", "Near Threshold", "Well-served"]
)
total_evs = tier_ev.sum()
print("\nEV registrations in each tier:")
for tier, evs in tier_ev.items():
    print(f"{tier}: {evs:.0f} EVs ({evs/total_evs:.1%} of total in matched cities)")

Coverage tier:
Under-served: 55 cities (23.8%)
Near Threshold: 32 cities (13.9%)
Well-served: 144 cities (62.3%)

EV registrations in each tier:
Under-served: 707986 EVs (86.0% of total in matched cities)
Near Threshold: 54532 EVs (6.6% of total in matched cities)
Well-served: 61034 EVs (7.4% of total in matched cities)


In [52]:
tier_color_scale = alt.Scale(
    domain=["Under-served", "Near Threshold", "Well-served"],
    range=["#CB392C", "#DD8452", "#55A868"],
)

tier_df = tier_summary.reset_index()
tier_df.columns = ["tier", "count"]
tier_df["pct"] = (tier_df["count"] / tier_df["count"].sum() * 100).round(1)

pie = (
    alt.Chart(tier_df)
    .mark_arc(outerRadius=110, innerRadius=40, stroke="white", strokeWidth=2)
    .encode(
        theta=alt.Theta("count:Q", stack=True),
        color=alt.Color(
            "tier:N",
            scale=tier_color_scale,
            legend=alt.Legend(
                title="Coverage Tier", orient="bottom",
                titleFontSize=11, labelFontSize=10,
            ),
        ),
        tooltip=[
            alt.Tooltip("tier:N", title="Tier"),
            alt.Tooltip("count:Q", title="Cities"),
            alt.Tooltip("pct:Q", title="Share (%)", format=".1f"),
        ],
    )
    .properties(
        title=alt.TitleParams(
            "Cities by Coverage Tier",
            subtitle="DC fast charger-to-EV ratio",
            fontSize=13, subtitleFontSize=10, subtitleColor="#666",
        ),
        width=240, height=240,
    )
)

scatter_df = df_scored[["location", "vehicle count", "fast charger ratio", "tier"]].copy()

scatter_chart = (
    alt.Chart(scatter_df)
    .mark_circle(opacity=0.72, size=50)
    .encode(
        x=alt.X(
            "vehicle count:Q",
            scale=alt.Scale(type="log"),
            title="EV Registrations (log scale)",
            axis=alt.Axis(grid=True, gridColor="#eee"),
        ),
        y=alt.Y(
            "fast charger ratio:Q",
            title="DC Fast Charger-to-EV Ratio",
            axis=alt.Axis(grid=True, gridColor="#eee", format=".4f"),
        ),
        color=alt.Color("tier:N", scale=tier_color_scale, legend=None),
        tooltip=[
            alt.Tooltip("location:N", title="City"),
            alt.Tooltip("tier:N", title="Tier"),
            alt.Tooltip("vehicle count:Q", title="EVs", format=","),
            alt.Tooltip("fast charger ratio:Q", title="Fast Ratio", format=".5f"),
        ],
    )
    .properties(
        title=alt.TitleParams(
            "Coverage Ratio vs. EV Demand by Coverage Tier",
            fontSize=13,
        ),
        width=380, height=280,
    )
)

fig1 = (
    alt.hconcat(pie, scatter_chart, spacing=30)
    .resolve_scale(color="shared")
    .configure_view(strokeWidth=0)
    .configure_axis(labelFontSize=10, titleFontSize=11)
)

fig1.save("docs/img/fig_coverage_tiers.png")
fig1


alt.HConcatChart(...)

## Step 3 — Priority Scoring

The coverage tier tells us *whether* a city is under-served. The priority score tells us *which* under-served cities to act on first.

Based on the team's analysis framing, the score combines three signals:

```
Priority = 0.5 * NeedScore + 0.3 * DemandScore + 0.2 * SpatialScore
```

| Component | Formula | Rationale |
|---|---|---|
| **NeedScore** | `1 / (ratio + e)`, normalised 0–1 | Lower ratio -> higher urgency |
| **DemandScore** | `log10(1 + vehicle_count)`, normalised 0–1 | Larger EV fleets generate more revenue and justify capital |
| **SpatialScore** | `max(0, 1 - NNI)`, normalised 0–1 | NNI < 1 -> chargers are clustered, leaving coverage gaps |

All three components are normalised to [0, 1] before combining. The final score is scaled to 0–100.

**Weights rationale:** Charger-to-EV ratio captures the supply–demand gap (primary signal per team note). Vehicle count captures commercial viability (secondary). NNI captures spatial quality of existing coverage (tertiary).


In [53]:
def minmax_norm(series):
    lo, hi = series.min(), series.max()
    if hi == lo:
        return pd.Series(0.5, index=series.index)
    return (series - lo) / (hi - lo)

EPSILON = 1e-6

df_scored["need_raw"]    = 1 / (df_scored["fast charger ratio"] + EPSILON)
df_scored["need_score"]  = minmax_norm(df_scored["need_raw"])

df_scored["demand_raw"]  = np.log10(1 + df_scored["vehicle count"])
df_scored["demand_score"] = minmax_norm(df_scored["demand_raw"])

df_scored["spatial_raw"]   = (1 - df_scored["NNI Index"]).clip(lower=0)
df_scored["spatial_score"] = minmax_norm(df_scored["spatial_raw"])

W_NEED, W_DEMAND, W_SPATIAL = 0.50, 0.30, 0.20
df_scored["priority_raw"] = (
    W_NEED    * df_scored["need_score"]
    + W_DEMAND  * df_scored["demand_score"]
    + W_SPATIAL * df_scored["spatial_score"]
)
df_scored["priority_score"] = (df_scored["priority_raw"] * 100).round(1)

print("Priority score distribution:")
print(df_scored["priority_score"].describe().round(2))

Priority score distribution:
count    231.00
mean      18.78
std       14.23
min        0.00
25%        9.90
50%       15.00
75%       23.15
max       75.10
Name: priority_score, dtype: float64


## Step 4: Ranked City List

In [54]:
DISPLAY_COLS = ["location", "tier", "ev_dc_fast_count", "vehicle count", "fast charger ratio", "NNI Index", "need_score", "demand_score", "spatial_score", "priority_score",]

ranked = (
    df_scored[DISPLAY_COLS]
    .sort_values("priority_score", ascending=False)
    .reset_index(drop=True)
)
ranked.index += 1

ranked["fast charger ratio"] = ranked["fast charger ratio"].map("{:.5f}".format)
ranked["NNI Index"]          = ranked["NNI Index"].map("{:.3f}".format)

print("Top 25 Priority Cities for Fast-Charger Expansion:")
print(ranked.head(25).to_string())

ranked.to_csv("city_priority_rankings.csv")

Top 25 Priority Cities for Fast-Charger Expansion:
           location          tier  ev_dc_fast_count  vehicle count fast charger ratio NNI Index  need_score  demand_score  spatial_score  priority_score
1     Richmond Hill  Under-served                 4        16646.0            0.00024     7.139    1.000000      0.835846       0.000000            75.1
2    West Vancouver  Under-served                 2         8130.0            0.00025    15.792    0.976902      0.766376       0.000000            71.8
3        Port Moody  Under-served                 1         3944.0            0.00025     1.000    0.947934      0.696259       0.000000            68.3
4   New Westminster  Under-served                 2         6206.0            0.00032     1.000    0.746401      0.740199       0.000000            59.5
5    Port Coquitlam  Under-served                 2         6182.0            0.00032     1.000    0.743523      0.739824       0.000000            59.4
6           Vaughan  Under-serv

In [56]:
ranked_plot = (
    df_scored[[
        "location", "tier", "ev_dc_fast_count", "vehicle count",
        "fast charger ratio", "NNI Index",
        "need_score", "demand_score", "spatial_score", "priority_score",
    ]]
    .sort_values("priority_score", ascending=False)
    .reset_index(drop=True)
)
top20 = ranked_plot.head(20).copy()

top20["need_contrib"]    = (top20["need_score"]    * W_NEED    * 100).round(2)
top20["demand_contrib"]  = (top20["demand_score"]  * W_DEMAND  * 100).round(2)
top20["spatial_contrib"] = (top20["spatial_score"] * W_SPATIAL * 100).round(2)

location_order = top20["location"].tolist()

top20_melted = top20.melt(
    id_vars=["location", "priority_score"],
    value_vars=["need_contrib", "demand_contrib", "spatial_contrib"],
    var_name="component",
    value_name="score_contribution",
)
component_labels = {
    "need_contrib":    f"Need ({int(W_NEED*100)}%)",
    "demand_contrib":  f"Demand ({int(W_DEMAND*100)}%)",
    "spatial_contrib": f"Spatial ({int(W_SPATIAL*100)}%)",
}
top20_melted["component"] = top20_melted["component"].map(component_labels)

component_order = [
    f"Need ({int(W_NEED*100)}%)",
    f"Demand ({int(W_DEMAND*100)}%)",
    f"Spatial ({int(W_SPATIAL*100)}%)",
]
component_color_scale = alt.Scale(
    domain=component_order,
    range=["#C44E52", "#4C72B0", "#8172B2"],
)

fig2 = (
    alt.Chart(top20_melted)
    .mark_bar()
    .encode(
        y=alt.Y(
            "location:N",
            sort=location_order,
            title=None,
            axis=alt.Axis(labelFontSize=9),
        ),
        x=alt.X("score_contribution:Q", title="Priority Score (0–100)", stack="zero"),
        color=alt.Color(
            "component:N",
            scale=component_color_scale,
            sort=component_order,
            legend=alt.Legend(title="Component"),
        ),
        order=alt.Order("component:N", sort="ascending"),
        tooltip=[
            "location:N",
            "component:N",
            alt.Tooltip("score_contribution:Q", format=".1f"),
            alt.Tooltip("priority_score:Q", format=".1f", title="Total Score"),
        ],
    )
    .properties(
        title="Top 20 Priority Cities — Score Breakdown",
        width=400,
        height=460,
    )
    .configure_view(strokeWidth=0)
    .configure_title(fontSize=13, fontWeight="bold")
    .configure_axis(labelFontSize=10, titleFontSize=11)
)

fig2.save("docs/img/fig_priority_ranking.png")
print("Saved docs/img/fig_priority_ranking.png")
fig2


Saved docs/img/fig_priority_ranking.png


alt.Chart(...)

## Step 5: High-Demand Under-Served Cities

The most commercially attractive targets are cities that are both under the 1/300 threshold and have large EV fleets

In [62]:
high_demand_underserved = (
    ranked_plot[ranked_plot["tier"] == "Under-served"]
    .sort_values("vehicle count", ascending=False)
    .head(15)
    .copy()
)
high_demand_underserved["annotation"] = high_demand_underserved.apply(
    lambda r: f"ratio={r['fast charger ratio']:.5f} | score={r['priority_score']:.0f}",
    axis=1,
)

location_order_u = high_demand_underserved["location"].tolist()

bars = (
    alt.Chart(high_demand_underserved)
    .mark_bar(color="#4C7BB5", opacity=0.85)
    .encode(
        y=alt.Y("location:N", sort=location_order_u, title=None),
        x=alt.X("vehicle count:Q", title="EV Registrations"),
        tooltip=[
            "location:N",
            alt.Tooltip("vehicle count:Q", format=","),
            alt.Tooltip("fast charger ratio:Q", format=".5f"),
            alt.Tooltip("priority_score:Q", format=".0f", title="Priority Score"),
        ],
    )
)

text_labels = (
    alt.Chart(high_demand_underserved)
    .mark_text(align="left", dx=6, fontSize=8, color="#444")
    .encode(
        y=alt.Y("location:N", sort=location_order_u),
        x=alt.X("vehicle count:Q"),
        text="annotation:N",
    )
)

fig3 = (
    (bars + text_labels)
    .properties(
        title=(
            "Top 15 Under-served Cities by EV Fleet Size "
            "(fast charger-EV ratio below 1/300 threshold)"
        ),
        width=500,
        height=360,
    )
    .configure_view(strokeWidth=0)
    .configure_title(fontSize=13, fontWeight="bold")
)

fig3.save("docs/img/fig_underserved_high_demand.png")
display(fig3)

print(
    high_demand_underserved[
        ["location", "ev_dc_fast_count", "vehicle count",
         "fast charger ratio", "NNI Index", "priority_score"]
    ].to_string(index=False)
)


alt.LayerChart(...)

       location  ev_dc_fast_count  vehicle count  fast charger ratio  NNI Index  priority_score
        Toronto                78        90504.0            0.000862   0.689449            53.4
      Vancouver                92        87968.0            0.001046   0.415554            59.1
         Surrey                47        61114.0            0.000769   1.172824            44.5
         Ottawa                30        38092.0            0.000788   0.728966            51.0
    Mississauga                64        33034.0            0.001937   0.522157            47.7
       Richmond                33        31648.0            0.001043   0.732024            46.6
        Burnaby                13        25294.0            0.000514   2.216329            49.7
        Markham                51        23582.0            0.002163   1.227610            31.7
       Brampton                12        23350.0            0.000514   1.297024            49.5
       Oakville                14       

## Next Steps (Week 4)

The following work is planned for the final week to complete the project deliverables:

### Analysis to complete
1. **Corridor analysis** — Extend coverage scoring from individual cities to inter-city travel corridors. Define corridors as pairs of major cities and identify highway segments with insufficient fast-charging coverage for a 400 km range EV. This directly addresses the proposal's objective of prioritising *travel corridors*.

2. **City name matching improvement** — Apply fuzzy string matching (e.g. `rapidfuzz`) to resolve the ~2,670 unmatched locations between the charger dataset and the vehicle registration dataset, improving coverage of medium-sized cities and towns.

3. **Sensitivity analysis on scoring weights** — Test alternative weight combinations (e.g. equal weights, higher spatial weight) to assess how stable the top-10 ranking is. This gives the planning audience confidence that the ranked list is robust, not an artefact of one weight choice.

4. **Interactive dashboard** — Build a `plotly` or `folium`-based interactive map combining the geographic station layer with the priority scores, allowing the planning team to filter by tier, province, or score threshold.

### Writing to complete
5. **Final planning brief** — A concise 2–3 page document targeted at a non-technical director of network planning. Will include: the ranked top-10 city list with rationale, 2–3 key maps, limitations, and recommended next actions.

6. **Blog post (Assignment 3)** — A data-science-audience write-up covering methodology choices (scoring formula, threshold selection, NNI as a spatial metric), reproducibility notes, and limitations. Will reference this notebook and the EDA notebook.

### Data quality note to address
> The current scoring cannot distinguish between a city with no chargers (true gap) and a city whose chargers simply didn't match to the vehicle registration data (a data artefact). The fuzzy matching work above is the primary mitigation.